# Dunnhumby semi-synthetic calibration — XGBoost

**Paper:** *Quantifying the Revenue Degradation of Decoupled Recommendation and Promotion Systems: A Joint Causal Framework*  
**Author:** Jagruthi Nalajala

This notebook calibrates a **semi-synthetic** response surface using real
Dunnhumby household covariates, category spending profiles, and coupon
redemption behavior. It is not a validation on observed causal outcomes:
potential outcomes are simulated after calibration. Each bootstrap replicate
trains on sampled households and evaluates learned policies only on its
disjoint out-of-bag households.

The learned joint and decoupled policies both use the same locked XGBoost
specification. The notebook reports whether it uses CUDA or CPU. The bootstrap
design and all data construction remain unchanged.


In [1]:
# XGBoost dependency and imports
import sys
import subprocess
import os
import warnings

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost>=2.0.0', '--quiet'])
    import xgboost as xgb

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from scipy.special import softmax, expit
from xgboost import XGBRegressor

print(f'XGBoost version: {xgb.__version__}')

required = [
    "transaction_data.csv", "hh_demographic.csv", "product.csv",
    "coupon.csv", "coupon_redempt.csv", "campaign_desc.csv",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. "
        "Download the Dunnhumby Complete Journey source files and place them "
        "in the same folder as this notebook."
    )
print("All required files found.")


XGBoost version: 3.2.0
All required files found.


In [2]:
# Select CUDA when it is genuinely available; otherwise use CPU explicitly.
# This changes compute hardware only—not the XGBoost model or experimental design.
import json

def detect_xgb_device():
    try:
        probe = XGBRegressor(
            objective='reg:squarederror', n_estimators=1,
            tree_method='hist', device='cuda', random_state=0,
            n_jobs=1, verbosity=0,
        )
        probe.fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))
        fitted_device = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        del probe
        if fitted_device == 'cuda':
            print('XGBoost execution device: CUDA GPU')
            return 'cuda'
        print('XGBoost execution device: CPU (CUDA is not exposed to this kernel)')
    except Exception as exc:
        print(f'XGBoost execution device: CPU (CUDA probe unavailable: {exc})')
    return 'cpu'

XGB_DEVICE = detect_xgb_device()


XGBoost execution device: CPU (CUDA is not exposed to this kernel)


In [3]:
# Locked learned-policy specification used in every experiment.
# Do not tune this configuration after observing the final comparison results.
def make_xgb(seed):
    return XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=10,
        subsample=0.90,
        colsample_bytree=0.90,
        tree_method='hist',
        device=XGB_DEVICE,
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )



In [4]:
# =============================================================
# Section 5.3: Load Dunnhumby data
# =============================================================

transactions = pd.read_csv('transaction_data.csv')
demographics = pd.read_csv('hh_demographic.csv')
coupons      = pd.read_csv('coupon.csv')
redemptions  = pd.read_csv('coupon_redempt.csv')
products     = pd.read_csv('product.csv')
campaigns    = pd.read_csv('campaign_desc.csv')

print(f'Transactions: {transactions.shape}')
print(f'Households with demographics: {demographics["household_key"].nunique()}')

Transactions: (2595732, 12)
Households with demographics: 801


In [5]:
# =============================================================
# Section 5.3: Build real customer features
# Features are derived from observed transactions and demographics.
# The top 10 departments by total spend serve as the R=10
# product categories used in the simulation.
# =============================================================

# Merge product department information
trans_prod = transactions.merge(
    products[['PRODUCT_ID', 'DEPARTMENT']], on='PRODUCT_ID', how='left')

# Keep top 10 departments as R=10 product categories
top_depts  = (trans_prod.groupby('DEPARTMENT')['SALES_VALUE']
              .sum().nlargest(10).index.tolist())
trans_top  = trans_prod[trans_prod['DEPARTMENT'].isin(top_depts)].copy()
dept_to_idx = {d: i for i, d in enumerate(top_depts)}
trans_top['dept_idx'] = trans_top['DEPARTMENT'].map(dept_to_idx)

print(f'Top 10 departments: {top_depts}')

# Restrict to households with demographic records (801 households)
hh_with_demo = demographics['household_key'].unique()
trans_top    = trans_top[trans_top['household_key'].isin(hh_with_demo)]
households   = sorted(trans_top['household_key'].unique())
print(f'Households with complete records: {len(households)}')

# Feature 1: Total spend in last 90 days (days 621-711)
recent      = trans_top[trans_top['DAY'] >= 621]
total_spend = (recent.groupby('household_key')['SALES_VALUE']
               .sum().reindex(households, fill_value=0))

# Feature 2: Purchase frequency (baskets per week over 102 weeks)
freq        = (trans_top.groupby('household_key')['BASKET_ID']
               .nunique() / 102).reindex(households, fill_value=0)

# Feature 3: Category breadth (distinct departments purchased)
breadth     = (trans_top.groupby('household_key')['dept_idx']
               .nunique().reindex(households, fill_value=0))

# Feature 4: Average basket size
basket      = (trans_top.groupby(['household_key', 'BASKET_ID'])['SALES_VALUE']
               .sum().reset_index()
               .groupby('household_key')['SALES_VALUE']
               .mean().reindex(households, fill_value=0))

# Feature 5: Coupon redemption rate
redempt_hh    = (redemptions.groupby('household_key')
                 .size().reindex(households, fill_value=0))
total_baskets = (trans_top.groupby('household_key')['BASKET_ID']
                 .nunique().reindex(households, fill_value=0))
redeem_rate   = redempt_hh / (total_baskets + 1)

# Features 6-10: Demographic variables
demo_enc = demographics.set_index('household_key')
age_map    = {'19-24':0,'25-34':1,'35-44':2,'45-54':3,'55-64':4,'65+':5}
income_map = {'Under 15K':0,'15-24K':1,'25-34K':2,'35-49K':3,
              '50-74K':4,'75-99K':5,'100-124K':6,'125-149K':7,
              '150-174K':8,'175-199K':9,'200-249K':10,'250K+':11}
hh_size_map = {'1':1,'2':2,'3':3,'4':4,'5+':5}

demo_feats             = pd.DataFrame(index=households)
demo_feats['age']      = demo_enc.loc[households,'AGE_DESC'].map(age_map).fillna(2)
demo_feats['income']   = demo_enc.loc[households,'INCOME_DESC'].map(income_map).fillna(3)
demo_feats['hh_size']  = demo_enc.loc[households,'HOUSEHOLD_SIZE_DESC'].map(hh_size_map).fillna(2)
demo_feats['married']  = (demo_enc.loc[households,'MARITAL_STATUS_CODE']=='A').astype(float)
demo_feats['homeowner']= (demo_enc.loc[households,'HOMEOWNER_DESC']=='Homeowner').astype(float)

# Assemble standardized feature matrix X_real (801 x 10)
def standardize(v):
    return (v - v.mean()) / (v.std() + 1e-6)

X_real = np.column_stack([
    standardize(total_spend.values),
    standardize(freq.values),
    standardize(breadth.values),
    standardize(basket.values),
    standardize(redeem_rate.values),
    demo_feats[['age','income','hh_size','married','homeowner']].values,
])

print(f'Feature matrix X_real: {X_real.shape}')

Top 10 departments: ['GROCERY', 'DRUG GM', 'PRODUCE', 'MEAT', 'KIOSK-GAS', 'MEAT-PCKGD', 'DELI', 'PASTRY', 'MISC SALES TRAN', 'NUTRITION']
Households with complete records: 801
Feature matrix X_real: (801, 10)


In [6]:
# =============================================================
# Section 5.3: DGP calibrated to real Dunnhumby spending patterns
# Baseline revenue is set from observed weekly category spend.
# Customer affinity scores are derived from real spending shares.
# Price sensitivity is estimated from observed coupon redemption rates.
# Discount depths reflect real observed coupon depths in the data.
# =============================================================

R_real = 10
O_real = 3
N_real = X_real.shape[0]

# Real mean spend per category per 2 years converted to weekly
# (dataset spans approximately 102 weeks)
dept_spend_arr = (trans_top
                  .groupby(['household_key','DEPARTMENT'])['SALES_VALUE']
                  .sum().unstack(fill_value=0)
                  .reindex(households, fill_value=0)[top_depts].values)

REAL_CATEGORY_SPEND = dept_spend_arr.mean(axis=0) / 102.0
print('Real weekly spend per category (mean across households):')
for dept, spend in zip(top_depts, REAL_CATEGORY_SPEND):
    print(f'  {dept}: ${spend:.2f}/week')

# Discount depths calibrated to real coupon data
# (mean coupon=$1.17, conservative fractions of category spend)
DISCOUNT_REAL = np.array([0.0, 0.05, 0.10])
SIGMA_REAL    = 0.15 * REAL_CATEGORY_SPEND.mean()

# Affinity from real department spend shares per household
AFF_real = dept_spend_arr / (dept_spend_arr.sum(axis=1, keepdims=True) + 1e-6)

# Price sensitivity from normalized coupon redemption rate
PS_real  = expit(standardize(redeem_rate.values))

# Baseline revenue = real weekly category spend per household
MU_real  = dept_spend_arr / 102.0

# Coupling matrix -- same non-separable design as primary simulation
# (rows sum to zero; RandomState(42) for Dunnhumby tier)
rng_real    = np.random.RandomState(42)
W_coup_real = rng_real.randn(R_real, O_real)
W_coup_real = W_coup_real - W_coup_real.mean(axis=1, keepdims=True)

assert np.allclose(W_coup_real.sum(axis=1), 0, atol=1e-10)
print(f'\nSIGMA_REAL: ${SIGMA_REAL:.3f}/week')
print(f'Discount depths: {DISCOUNT_REAL}')

Real weekly spend per category (mean across households):
  GROCERY: $27.66/week
  DRUG GM: $7.19/week
  PRODUCE: $4.01/week
  MEAT: $3.59/week
  KIOSK-GAS: $4.19/week
  MEAT-PCKGD: $2.56/week
  DELI: $1.81/week
  PASTRY: $0.80/week
  MISC SALES TRAN: $0.87/week
  NUTRITION: $0.74/week

SIGMA_REAL: $0.801/week
Discount depths: [0.   0.05 0.1 ]


In [7]:
# =============================================================
# DGP simulator using calibrated Dunnhumby parameters
# Equation 21 in paper with real-data-calibrated mu, AFF, PS.
# =============================================================

def simulate_dunnhumby(X_sub, AFF_sub, PS_sub, MU_sub, lambda_val, seed):
    """
    Simulate potential outcomes using the Dunnhumby-calibrated response surface.
    Baseline revenue mu(X, r) comes from real weekly category spend.
    Affinity AFF_sub comes from real household spending shares.
    Price sensitivity PS_sub is estimated from real coupon redemption rates.
    Discount depths DISCOUNT_REAL reflect real observed coupon depths.
    """
    rng_y = np.random.RandomState(seed)
    N_sub   = len(X_sub)
    Y       = np.zeros((N_sub, R_real, O_real))
    mu_mean = MU_sub.mean(axis=1)
    for r in range(R_real):
        for o in range(O_real):
            base  = MU_sub[:, r] + PS_sub * DISCOUNT_REAL[o] * mu_mean
            gamma = AFF_sub[:, r] * MU_sub[:, r] * W_coup_real[r, o] * PS_sub
            eps   = rng_y.normal(0, SIGMA_REAL, N_sub)
            Y[:, r, o] = np.maximum(base + lambda_val * gamma + eps, 0)
    return Y

# Validate oracle gap before running full bootstrap
print('Oracle gap validation on real Dunnhumby customers:')
for lam in [0.0, 0.25, 0.50, 0.75, 1.0]:
    Y_l  = simulate_dunnhumby(X_real, AFF_real, PS_real, MU_real, lam, seed=42)
    base = Y_l[:, 0, 0]
    incr = Y_l - base[:, None, None]
    flat = incr.reshape(N_real, -1)
    irc_o  = flat.max(axis=1).mean()
    tr2    = incr.mean(axis=2).argmax(axis=1)
    to2    = incr.mean(axis=1).argmax(axis=1)
    irc_do = incr[np.arange(N_real), tr2, to2].mean()
    d      = irc_o - irc_do
    print(f'  lambda={lam:.2f}: Oracle Delta=${d:.3f}/week (${d*52:.2f}/year)')


Oracle gap validation on real Dunnhumby customers:
  lambda=0.00: Oracle Delta=$0.424/week ($22.07/year)
  lambda=0.25: Oracle Delta=$0.397/week ($20.64/year)
  lambda=0.50: Oracle Delta=$0.413/week ($21.46/year)
  lambda=0.75: Oracle Delta=$0.427/week ($22.21/year)
  lambda=1.00: Oracle Delta=$0.503/week ($26.15/year)


In [8]:
# =============================================================
# Section 6.6: Out-of-bag bootstrap semi-synthetic calibration
# Each replicate draws training households with replacement and evaluates
# learned policies on its out-of-bag households. This is a genuine bootstrap
# design; no household appears in both train and test within a replicate.
# =============================================================

N_BOOTSTRAP = 500
TRAIN_FRACTION = 0.60
MIN_OOB_HOUSEHOLDS = 100
LAMBDA_GRID = [0.0, 0.25, 0.50, 0.75, 1.0]
BETA_BASE = 0.50
all_rows = []

print(f"Starting OOB bootstrap: {N_BOOTSTRAP} replicates × {len(LAMBDA_GRID)} lambda values")
print(f"Panel size: {N_real} households; training draws per replicate: {int(TRAIN_FRACTION * N_real)}")
print()

for b in range(N_BOOTSTRAP):
    rng_b = np.random.RandomState(10_000 + b)

    # Bootstrap training sample: observations may be repeated.
    n_train_draws = int(TRAIN_FRACTION * N_real)
    train_idx = rng_b.choice(N_real, size=n_train_draws, replace=True)
    in_bag = np.zeros(N_real, dtype=bool)
    in_bag[np.unique(train_idx)] = True
    test_idx = np.flatnonzero(~in_bag)  # OOB households, disjoint from training

    if len(test_idx) < MIN_OOB_HOUSEHOLDS:
        raise RuntimeError(f"Bootstrap replicate {b} has too few OOB households: {len(test_idx)}")

    n_tr, n_te = len(train_idx), len(test_idx)
    X_tr, X_te = X_real[train_idx], X_real[test_idx]

    for lam in LAMBDA_GRID:
        # Simulate one full outcome matrix per replicate before subsetting.
        # Repeated training draws therefore retain the same simulated response,
        # as required for a bootstrap resample of a fixed synthetic dataset.
        y_all = simulate_dunnhumby(
            X_real, AFF_real, PS_real, MU_real, lam,
            seed=100_000 + b * 100 + int(lam * 10),
        )
        y_tr, y_te = y_all[train_idx], y_all[test_idx]

        base_tr = y_tr[:, 0, 0]
        incr_tr = y_tr - base_tr[:, None, None]
        flat_tr = incr_tr.reshape(n_tr, -1)
        tau_rec_tr = incr_tr.mean(axis=2)
        tau_off_tr = incr_tr.mean(axis=1)

        base_te = y_te[:, 0, 0]
        incr_te = y_te - base_te[:, None, None]
        flat_te = incr_te.reshape(n_te, -1)
        k = max(1, int(BETA_BASE * n_te))

        # Structural semi-synthetic benchmark: full outcomes are available only
        # because the response surface is simulated. This is not a logged-data learner.
        joint_models = [
            make_xgb(1_000 * b + a).fit(X_tr, flat_tr[:, a])
            for a in range(R_real * O_real)
        ]
        rec_models = [
            make_xgb(2_000 * b + r).fit(X_tr, tau_rec_tr[:, r])
            for r in range(R_real)
        ]
        offer_models = [
            make_xgb(3_000 * b + o).fit(X_tr, tau_off_tr[:, o])
            for o in range(O_real)
        ]

        joint_hat = np.column_stack([m.predict(X_te) for m in joint_models])
        rec_hat = np.column_stack([m.predict(X_te) for m in rec_models])
        offer_hat = np.column_stack([m.predict(X_te) for m in offer_models])

        # Evaluate each learned policy on the disjoint OOB set.
        best_joint = joint_hat.argmax(axis=1)
        selected_joint = np.zeros(n_te, dtype=bool)
        selected_joint[np.argsort(joint_hat.max(axis=1))[::-1][:k]] = True
        contribution_joint = np.where(
            selected_joint, flat_te[np.arange(n_te), best_joint], 0.0
        )

        best_rec = rec_hat.argmax(axis=1)
        best_offer = offer_hat.argmax(axis=1)
        selected_dec = np.zeros(n_te, dtype=bool)
        selected_dec[np.argsort(rec_hat.max(axis=1) + offer_hat.max(axis=1))[::-1][:k]] = True
        contribution_dec = np.where(
            selected_dec, incr_te[np.arange(n_te), best_rec, best_offer], 0.0
        )

        delta = (contribution_joint - contribution_dec).mean()
        all_rows.append({
            "bootstrap": b,
            "lambda": lam,
            "n_train_draws": n_tr,
            "n_train_unique": int(in_bag.sum()),
            "n_oob": n_te,
            "irc_joint_week": contribution_joint.mean(),
            "irc_decoupled_week": contribution_dec.mean(),
            "delta_week": delta,
            "delta_year": 52 * delta,
        })

    if (b + 1) % 10 == 0:
        df_so_far = pd.DataFrame(all_rows)
        current = df_so_far[df_so_far["lambda"] == 0.75]["delta_week"].mean()
        print(f"Replicate {b + 1}/{N_BOOTSTRAP} | lambda=0.75 mean OOB gap: ${current:.3f}/week")

    if (b + 1) % 50 == 0:
        pd.DataFrame(all_rows).to_csv("results_dunnhumby_partial.csv", index=False)

df_dunn = pd.DataFrame(all_rows)
df_dunn.to_csv("results_dunnhumby.csv", index=False)
print(f"\nSaved results_dunnhumby.csv ({len(df_dunn)} rows)")


Starting OOB bootstrap: 500 replicates × 5 lambda values
Panel size: 801 households; training draws per replicate: 480

Replicate 10/500 | lambda=0.75 mean OOB gap: $0.786/week
Replicate 20/500 | lambda=0.75 mean OOB gap: $0.804/week
Replicate 30/500 | lambda=0.75 mean OOB gap: $0.804/week
Replicate 40/500 | lambda=0.75 mean OOB gap: $0.808/week
Replicate 50/500 | lambda=0.75 mean OOB gap: $0.817/week
Replicate 60/500 | lambda=0.75 mean OOB gap: $0.810/week
Replicate 70/500 | lambda=0.75 mean OOB gap: $0.808/week
Replicate 80/500 | lambda=0.75 mean OOB gap: $0.813/week
Replicate 90/500 | lambda=0.75 mean OOB gap: $0.809/week
Replicate 100/500 | lambda=0.75 mean OOB gap: $0.809/week
Replicate 110/500 | lambda=0.75 mean OOB gap: $0.813/week
Replicate 120/500 | lambda=0.75 mean OOB gap: $0.814/week
Replicate 130/500 | lambda=0.75 mean OOB gap: $0.818/week
Replicate 140/500 | lambda=0.75 mean OOB gap: $0.817/week
Replicate 150/500 | lambda=0.75 mean OOB gap: $0.816/week
Replicate 160/500 |

In [9]:
# =============================================================
# Table 7: OOB-bootstrap semi-synthetic calibration results
# Percentile interval summarizes variation across 500 bootstrap replicates.
# =============================================================

print("Table 7: Dunnhumby semi-synthetic calibration (OOB bootstrap)")
print(f"({N_real} households, {N_BOOTSTRAP} bootstrap replicates)")
print()
print(f'{"lambda":<8} {"Gap/week":>12} {"SD/week":>10} '
      f'{"2.5%":>10} {"97.5%":>10} '
      f'{"Gap/year":>12} {"Interval > 0?":>15}')
print("-" * 90)

for lam in LAMBDA_GRID:
    d = df_dunn[df_dunn["lambda"] == lam]
    week_mean = d["delta_week"].mean()
    week_sd = d["delta_week"].std(ddof=1)
    low, high = np.percentile(d["delta_week"], [2.5, 97.5])
    year_mean = d["delta_year"].mean()
    positive = "Yes" if low > 0 else "No"
    print(f"{lam:<8.2f} "
          f"${week_mean:>9.3f} ${week_sd:>8.3f} "
          f"${low:>8.3f} ${high:>8.3f} "
          f"${year_mean:>9.2f} {positive:>15}")

print("\nInterpretation: this table is a semi-synthetic structural calibration study.")
print("It does not establish a causal effect from the observed Dunnhumby transactions.")


Table 7: Dunnhumby semi-synthetic calibration (OOB bootstrap)
(801 households, 500 bootstrap replicates)

lambda       Gap/week    SD/week       2.5%      97.5%     Gap/year   Interval > 0?
------------------------------------------------------------------------------------------
0.00     $    0.105 $   0.044 $   0.024 $   0.192 $     5.46             Yes
0.25     $    0.384 $   0.050 $   0.287 $   0.476 $    19.99             Yes
0.50     $    0.612 $   0.068 $   0.484 $   0.746 $    31.80             Yes
0.75     $    0.811 $   0.081 $   0.660 $   0.960 $    42.15             Yes
1.00     $    0.990 $   0.086 $   0.821 $   1.183 $    51.47             Yes

Interpretation: this table is a semi-synthetic structural calibration study.
It does not establish a causal effect from the observed Dunnhumby transactions.
